# Pipeline de automatización del reentrenamiento del modelo

Este notebook diseña una primera versión del pipeline de automatización del entrenamiento del modelo de predicción de tráfico del proyecto `Movilidad_inteligente_madrid`.

El objetivo es preparar un flujo capaz de:

1. Detectar nuevos datos de tráfico.
2. Registrar qué archivos ya han sido detectados.
3. Validar de forma básica los archivos disponibles.
4. Simular el proceso de reentrenamiento del modelo.
5. Generar logs de ejecución.

En esta fase inicial no se ejecuta todavía el entrenamiento real del LSTM, ya que el notebook original del modelo pertenece a una fuente externa y depende de archivos de datos pesados que no están incluidos en este repositorio.


In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import json


## 1. Configuración de rutas

Se definen las rutas principales del proyecto. Si se ejecuta desde Google Colab, la ruta base deberá adaptarse a la ubicación real del repositorio en Google Drive.


In [ ]:
# Cambiar esta ruta si el repositorio está guardado con otro nombre en Drive
PROJECT_ROOT = Path('/content/drive/MyDrive/Movilidad_inteligente_madrid-main')

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
AUTOMATIZACION_DIR = PROJECT_ROOT / 'automatizacion_modelo'
LOGS_DIR = AUTOMATIZACION_DIR / 'logs'
MANIFEST_PATH = AUTOMATIZACION_DIR / 'manifest_datos.csv'

AUTOMATIZACION_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print('Ruta del proyecto:', PROJECT_ROOT)
print('Carpeta de datos procesados:', PROCESSED_DIR)
print('Carpeta de automatización:', AUTOMATIZACION_DIR)
print('Carpeta de logs:', LOGS_DIR)


## 2. Comprobación de archivos disponibles

Se listan los archivos disponibles en la carpeta `data/processed`, que será la carpeta utilizada como ejemplo para detectar nuevos datos.


In [ ]:
if PROCESSED_DIR.exists():
    archivos = list(PROCESSED_DIR.glob('*'))
    print(f'Se han encontrado {len(archivos)} archivos en data/processed:')
    for archivo in archivos:
        print('-', archivo.name)
else:
    archivos = []
    print('La carpeta data/processed no existe todavía.')


## 3. Detección de nuevos datos

El pipeline mantiene un archivo `manifest_datos.csv` donde se registran los archivos ya detectados. Así se evita procesar el mismo archivo varias veces.


In [ ]:
def cargar_manifest():
    """Carga el registro de archivos ya detectados."""
    if MANIFEST_PATH.exists():
        return pd.read_csv(MANIFEST_PATH)
    
    return pd.DataFrame(columns=['archivo', 'ruta', 'fecha_detectado', 'extension'])


def detectar_nuevos_datos(carpeta_datos):
    """Detecta archivos nuevos en la carpeta indicada."""
    manifest = cargar_manifest()
    archivos_registrados = set(manifest['archivo'].tolist())

    if not carpeta_datos.exists():
        return [], manifest

    archivos_actuales = list(carpeta_datos.glob('*'))
    nuevos = []

    for archivo in archivos_actuales:
        if archivo.is_file() and archivo.name not in archivos_registrados:
            nuevos.append({
                'archivo': archivo.name,
                'ruta': str(archivo),
                'fecha_detectado': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'extension': archivo.suffix
            })

    return nuevos, manifest


In [ ]:
nuevos, manifest = detectar_nuevos_datos(PROCESSED_DIR)

if nuevos:
    print(f'Se han detectado {len(nuevos)} archivos nuevos:')
    display(pd.DataFrame(nuevos))
else:
    print('No se han detectado archivos nuevos.')


## 4. Actualización del manifest

Si se detectan archivos nuevos, se registran en el manifest para dejar constancia de la fecha de detección.


In [ ]:
def actualizar_manifest(nuevos, manifest):
    """Actualiza el archivo manifest con los nuevos datos detectados."""
    if not nuevos:
        print('No hay nuevos archivos que registrar.')
        return manifest

    df_nuevos = pd.DataFrame(nuevos)
    manifest_actualizado = pd.concat([manifest, df_nuevos], ignore_index=True)
    manifest_actualizado.to_csv(MANIFEST_PATH, index=False)

    print('Manifest actualizado correctamente.')
    return manifest_actualizado


manifest = actualizar_manifest(nuevos, manifest)
display(manifest)


## 5. Validación básica de archivos

Antes de incorporar archivos al proceso de entrenamiento, se comprueba que existen, no están vacíos y tienen una extensión esperada.


In [ ]:
def validar_archivo(archivo):
    """Realiza una validación básica de un archivo."""
    ruta = Path(archivo)

    if not ruta.exists():
        return False, 'El archivo no existe'

    if ruta.stat().st_size == 0:
        return False, 'El archivo está vacío'

    if ruta.suffix.lower() not in ['.csv', '.json', '.html', '.pkl', '.joblib']:
        return False, 'Extensión no esperada'

    return True, 'Archivo válido'


resultados_validacion = []

if PROCESSED_DIR.exists():
    for archivo in PROCESSED_DIR.glob('*'):
        if archivo.is_file():
            valido, mensaje = validar_archivo(archivo)
            resultados_validacion.append({
                'archivo': archivo.name,
                'valido': valido,
                'mensaje': mensaje
            })

df_validacion_archivos = pd.DataFrame(resultados_validacion)
display(df_validacion_archivos)


## 6. Pipeline de reentrenamiento simulado

Se define la estructura general del pipeline. En esta versión inicial el entrenamiento real del LSTM no se ejecuta, pero queda indicada la fase en la que se integraría.


In [ ]:
def preparar_datos():
    print('1. Preparando datos para entrenamiento...')
    print('   En esta fase se integrarían los nuevos datos al histórico.')


def entrenar_modelo():
    print('2. Entrenando modelo LSTM...')
    print('   Entrenamiento real pendiente de integración con el notebook externo.')


def evaluar_modelo():
    print('3. Evaluando modelo...')
    print('   Se calcularían métricas como RMSE, MAE y error porcentual.')


def versionar_modelo():
    print('4. Versionando modelo...')
    print('   Si el modelo mejora, se guardaría una nueva versión.')


def ejecutar_pipeline():
    print('Inicio del pipeline:', datetime.now())
    print('-' * 60)

    preparar_datos()
    entrenar_modelo()
    evaluar_modelo()
    versionar_modelo()

    print('-' * 60)
    print('Fin del pipeline:', datetime.now())


ejecutar_pipeline()


## 7. Registro de ejecución

Cada ejecución del pipeline genera un log en formato JSON para poder consultar posteriormente qué ocurrió.


In [ ]:
log = {
    'fecha_ejecucion': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'estado': 'ejecucion_simulada',
    'entrenamiento_lstm': 'no_ejecutado',
    'motivo': 'El entrenamiento real depende de datos pesados no incluidos en GitHub',
    'archivos_detectados': len(nuevos),
    'archivos_validados': len(df_validacion_archivos)
}

log_path = LOGS_DIR / f"log_pipeline_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

with open(log_path, 'w', encoding='utf-8') as f:
    json.dump(log, f, indent=4, ensure_ascii=False)

print('Log guardado en:', log_path)


## Conclusión

En este notebook se ha diseñado una primera versión del pipeline de automatización del entrenamiento del modelo de predicción de tráfico.

Aunque todavía no se ejecuta el entrenamiento real del LSTM, se ha preparado la estructura básica del proceso:

- detección de nuevos datos;
- registro de archivos detectados;
- validación básica de archivos;
- simulación del flujo de reentrenamiento;
- generación de logs de ejecución.

Esta estructura permitirá integrar posteriormente el notebook externo del modelo LSTM cuando los datos de entrenamiento estén disponibles en una ruta accesible.

La separación entre código y datos permite mantener el repositorio ligero, evitando subir a GitHub archivos pesados como históricos de tráfico o modelos entrenados.
